# Colab batch jobs — job queue notebook

Ready-to-run job cells for a **T4 runtime** (Runtime → Change runtime type → T4).
Pattern per `docs/kb/colab-mcp-jobs.md`: setup once → run job cells → Devin
collects outputs and lands them durably (CMS reports / card comms).

- **Setup cell** installs ollama + pulls `gemma3:12b` + `llama3.2:3b`
  (~3 min on a fresh runtime; needed by Jobs 2–3).
- **Job cells are independent** — run any subset, in any order.
- Devin can drive this whole notebook via the `colab` MCP and refreshes the
  `# CARDS:` data snapshot in Job 2 before running it.

## Setup — ollama + models (run once)

In [ ]:
# ---- install ollama + pull models (needed by Jobs 2-3) ----
import subprocess, os, time, urllib.request

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)

os.makedirs('/tmp/ollama/bin', exist_ok=True)
if sh('test -x /tmp/ollama/bin/ollama').returncode != 0:
    r = sh('apt-get install -y zstd >/dev/null 2>&1; '
           'curl -sL https://ollama.com/download/ollama-linux-amd64.tar.zst '
           '| tar --use-compress-program=unzstd -x -C /tmp/ollama')
    print('install rc:', r.returncode)

r = sh('/tmp/ollama/bin/ollama --version')
print(r.stdout.strip() or r.stderr.strip())

subprocess.Popen(['/tmp/ollama/bin/ollama','serve'],
    stdout=open('/tmp/ollama-serve.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=1); break
    except Exception: time.sleep(1)
print('ollama up:', urllib.request.urlopen('http://127.0.0.1:11434/api/version').read().decode())

for model in ['gemma3:12b', 'llama3.2:3b']:
    t = time.time()
    r = sh(f'/tmp/ollama/bin/ollama pull {model}', timeout=1200)
    print(f'{model}: pulled in {time.time()-t:.0f}s rc={r.returncode}')
print('SETUP-OK')

## Job 1 — fleet download benchmark (no model needed)

Measures Colab's real throughput to the sources we actually pull from.
Prints a JSON row per source. Devin compares against fleet numbers
(dell→brainchip 9.7MB/s, dell→gdrive 13.5MiB/s verified 2026-10-12) and
lands the comparison in the `colab-gdrive-cache` KB.

In [ ]:
# ---- Job 1: download benchmark ----
import time, json, urllib.request

def bench(name, url, cap_mb=80):
    t = time.time(); got = 0
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            while True:
                chunk = r.read(1<<20)
                if not chunk: break
                got += len(chunk)
                if time.time()-t > 60 or got >= cap_mb<<20: break
        dt = time.time()-t
        return {'source': name, 'mb': round(got/1e6,1), 's': round(dt,1),
                'mbps': round(got/dt/1e6,2)}
    except Exception as e:
        return {'source': name, 'error': str(e)[:80]}

results = [
    bench('brainchip-cifar10', 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'),
    bench('hf-cdn', 'https://huggingface.co/datasets/mnist/resolve/main/mnist/train-00000-of-00001.parquet'),
    bench('github-release', 'https://github.com/ollama/ollama/releases/download/v0.40.3/ollama-linux-amd64.tgz', 60),
    bench('toronto-cifar10', 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', 20),
]
print(json.dumps(results, indent=2))
print('JOB1-DONE')

## Job 2 — backlog cluster map (needs Setup)

`CARDS` = snapshot of open kanban cards (Devin refreshes via `update_cell`
before running). `gemma3:12b` clusters them into named work-streams —
feeds board reorganization + the triage review. Output: JSON per batch —
Devin merges and lands it as a CMS leaf report under
`backlog-triage-2026-10-12`.

In [ ]:
# ---- Job 2: cluster open cards into work-streams ----
# Devin refreshes this variable at run time via MCP update_cell.
CARDS = '''
PASTE-CARDS-HERE — Devin injects the live snapshot before running.
'''.strip()

import json, urllib.request, concurrent.futures, time

def ask(prompt, model='gemma3:12b'):
    body = json.dumps({'model': model, 'prompt': prompt, 'stream': False,
                       'options': {'temperature': 0.1, 'num_predict': 2000}}).encode()
    r = urllib.request.urlopen('http://127.0.0.1:11434/api/generate', body, timeout=600)
    return json.loads(r.read())['response']

BATCH = 40
cards = [l.strip() for l in CARDS.splitlines() if l.strip()]
prompts = []
for i in range(0, len(cards), BATCH):
    chunk = chr(10).join(cards[i:i+BATCH])
    prompts.append(
        'Cluster these kanban cards into 4-8 named work-streams (epics). '
        'Reply ONLY JSON: {"streams":[{"name":..,"cards":[..]}]} using the '
        'card ids verbatim.
' + chunk)

t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(2) as ex:
    outs = list(ex.map(ask, prompts))
print(f'{len(cards)} cards, {len(prompts)} batches, {time.time()-t0:.0f}s')
for o in outs: print(o[:1500])
print('JOB2-DONE')

## Job 3 — model-eval: gemma3:12b (T4) vs llama3.2:3b

The "what does the Colab lane buy us" benchmark: same 6-prompt suite on
both models — reasoning, JSON extraction, summarization, Thai, code,
classification. Reports tokens/sec + output side by side. Devin lands the
comparison as a CMS leaf report; feeds the gateway model-tiering
decision.

In [ ]:
# ---- Job 3: A/B model-eval ----
import json, urllib.request, time

SUITE = [
  ('reason',  'A farmer has 17 sheep; all but 9 die. How many remain? One line.'),
  ('json',    'Extract as JSON {"city":..,"country":..}: "Chaba HQ is in Sattahip, Thailand".'),
  ('summ',    'Summarize in one sentence: The fleet runs 8 hosts; idc03 carries Ada backends, mddb and liteLLM; omen is the GPU lane; dell is the kanban writer; macbook is the portable lab.'),
  ('thai',    'ตอบเป็นภาษาไทยสั้นๆ: รถติดมาก ฉันควรออกจากบ้านกี่โมงถ้าต้องไปถึง 18:00'),
  ('code',    'Write a one-line python function that returns the nth fibonacci number.'),
  ('class',   'Classify priority high|med|low, one word: "ada-ha hardening — stable private prod".'),
]

def run(model, prompt):
    t = time.time()
    body = json.dumps({'model': model, 'prompt': prompt, 'stream': False,
                       'options': {'temperature': 0.1, 'num_predict': 150}}).encode()
    r = json.loads(urllib.request.urlopen('http://127.0.0.1:11434/api/generate', body, timeout=300).read())
    dt = time.time()-t
    n = r.get('eval_count',0); d = r.get('eval_duration',1)/1e9 or dt
    return r['response'].strip()[:200], round(dt,1), round(n/d,1)

report = {}
for name, prompt in SUITE:
    row = {}
    for model in ['gemma3:12b','llama3.2:3b']:
        out, dt, tps = run(model, prompt)
        row[model] = {'out': out, 's': dt, 'tps': tps}
    report[name] = row
    print(f'== {name}')
    for m, v in row.items(): print(f'   {m}: {v["tps"]} tok/s | {v["out"][:90]}')
open('/tmp/model-eval.json','w').write(json.dumps(report, indent=2, ensure_ascii=False))
print('JOB3-DONE')